# Practical 10: Feature Scaling (StandardScaler vs MinMaxScaler) & Train-Test Split

**UNIT 3: Preparing Big Data for Machine Learning**

**Problem Statement:** Write a Python program to perform feature scaling and dataset partitioning. Apply StandardScaler (Z-score normalization) and MinMaxScaler to numerical features, compare their transformed distributions, and split the dataset into training (80%) and testing (20%) sets using train_test_split(..., test_size=0.2, random_state=42).

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

pd.set_option("display.width", 200)
rnd = np.random.default_rng(123)
n = 130
age = rnd.integers(24, 62, n).astype(float)
salary = np.round(30000 + age * 1010 + rnd.normal(0, 7500, n), -2)
tbl = pd.DataFrame({
    "Country": rnd.choice(["Greece", "Peru", "Nepal"], n),
    "Age": age,
    "Salary": salary,
    "Purchased": np.where(salary + rnd.normal(0, 8000, n) > 58000, "Yes", "No"),
})
tbl.loc[rnd.choice(n, 7, replace=False), "Age"] = np.nan
tbl.loc[rnd.choice(n, 7, replace=False), "Salary"] = np.nan
print(tbl.head(12))

num_cols = ["Age", "Salary"]
tbl[num_cols] = tbl[num_cols].fillna(tbl[num_cols].median())
print(tbl[num_cols].isna().sum())

std_df = pd.DataFrame(StandardScaler().fit_transform(tbl[num_cols]), columns=num_cols)
mm_df = pd.DataFrame(MinMaxScaler().fit_transform(tbl[num_cols]), columns=num_cols)

stats = ["mean", "std", "min", "max"]
print("=== Original ===")
print(tbl[num_cols].describe().loc[stats].round(3))
print("\n=== StandardScaler (mean ≈ 0, std ≈ 1) ===")
print(std_df.describe().loc[stats].round(3))
print("\n=== MinMaxScaler (range [0, 1]) ===")
print(mm_df.describe().loc[stats].round(3))

X = pd.get_dummies(tbl.drop(columns=["Purchased"]), columns=["Country"], dtype=int)
y = tbl["Purchased"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)
print(f"Total rows: {len(X)} | Train: {X_train.shape} | Test: {X_test.shape}")

fit_scaler = StandardScaler()
X_train_s, X_test_s = X_train.copy(), X_test.copy()
X_train_s[num_cols] = fit_scaler.fit_transform(X_train[num_cols])
X_test_s[num_cols] = fit_scaler.transform(X_test[num_cols])

print("Scaled TRAIN stats (mean ≈ 0, std ≈ 1):")
print(X_train_s[num_cols].describe().loc[["mean", "std"]].round(3))
print("\nScaled TEST stats (close to, but not exactly, 0/1 – as expected):")
print(X_test_s[num_cols].describe().loc[["mean", "std"]].round(3))

,Country,Age,Salary,Purchased
0,Nepal,24.0,50800.0,No
1,Greece,49.0,68700.0,No
2,Greece,46.0,86700.0,Yes
3,Greece,26.0,59600.0,No
4,Peru,58.0,83200.0,Yes
5,Greece,32.0,64500.0,Yes
6,Peru,33.0,60000.0,Yes
7,Nepal,31.0,59700.0,Yes
8,Peru,36.0,69100.0,Yes
9,Greece,30.0,67400.0,Yes


Age       0
Salary    0
dtype: int64
=== Original ===

,Age,Salary
mean,41.200,72942.308
std,10.062,11650.417
min,24.000,43100.000
max,59.000,100300.000



=== StandardScaler (mean ≈ 0, std ≈ 1) ===

,Age,Salary
mean,-0.000,0.000
std,1.004,1.004
min,-1.716,-2.571
max,1.776,2.357


...

,Age,Salary
mean,0.491,0.522
std,0.287,0.204
min,0.000,0.000
max,1.000,1.000


Total rows: 130 | Train: (104, 5) | Test: (26, 5)
Scaled TRAIN stats (mean ≈ 0, std ≈ 1):

,Age,Salary
mean,0.000,0.000
std,1.005,1.005



Scaled TEST stats (close to, but not exactly, 0/1 – as expected):

,Age,Salary
mean,0.028,0.077
std,1.078,1.130
